# Red Sox Next Year OPS Predictions

## Loading data with multiple yrs of OPS data

In [1]:
library(tidyverse)
data = read_csv("https://huggingface.co/spaces/rkarthur/sabr3evaluation/resolve/main/data/SABR3_FinalAssignment_data.csv")
head(data)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 175 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): bbref_id
dbl (15): Age, PA1, PA2, PA3, PA4, OPSY1, OPSY2, OPSY3, OPSY4, weighted_avg,...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


bbref_id,Age,PA1,PA2,PA3,PA4,OPSY1,OPSY2,OPSY3,OPSY4,weighted_avg,reliability,regressed_ops,age_adjusted_ops,ops_pred,ops_real
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
abreujo02,33,262,693,553,675,0.987,0.8340000,0.798,0.906,0.8887500,0.5568685,0.8139716,0.8042039,0.8042039,0.831
adriaeh01,30,101,236,366,186,0.557,0.7650000,0.680,0.707,0.6570833,0.3694167,0.6967575,0.6946673,0.6946673,0.728
aguilje01,30,216,369,566,311,0.809,0.7131382,0.890,0.837,0.7972961,0.4895789,0.7578425,0.7555690,0.7555690,0.788
ahmedni01,30,217,625,564,178,0.729,0.7530000,0.700,0.717,0.7297500,0.5395242,0.7252604,0.7230846,0.7230846,0.619
albieoz01,23,124,702,684,244,0.773,0.8520000,0.757,0.810,0.7953333,0.5571956,0.7619754,0.7345443,0.7345443,0.799
alfarjo01,27,100,465,377,114,0.624,0.7360000,0.731,0.874,0.6880833,0.4397759,0.7059638,0.6974923,0.6974923,0.625


## Load packages

In [2]:
if (!requireNamespace('devtools', quietly = TRUE)){
  install.packages('devtools')
}
devtools::install_github(repo = "BillPetti/baseballr")

Skipping install of 'baseballr' from a github remote, the SHA1 (ff996dec) has not changed since last install.
  Use `force = TRUE` to force installation



In [3]:
library(baseballr)

In [4]:
install.packages("RSQLite")
library(RSQLite)

Installing package into ‘/srv/r’
(as ‘lib’ is unspecified)



reliability = total PA / (total PA + 1200)
Regressed rate = (Player reliability * Player Rate) + ((1 – Player reliability) * league average)

The adjustment is applied as a simple multiplier. So, for a given age, your expected production either gets multiplied above 1 (increasing it) or below 1 (decreasing it). The multiplier looks like this:



## Create Hitting Consistency Feature

In [5]:
Lahman <- Lahman::Batting %>% filter((yearID > 2017) & (yearID < 2022))
Lahman <- Lahman[Lahman$playerID %in% data$bbref_id, ]

In [6]:
Lahman <- Lahman%>%mutate(overall_hitting = ((H + SF + BB +  IBB + HBP)/AB) - (SO/AB))
Lahman <- Lahman%>%group_by(playerID, yearID)%>%summarise(overall_hitting_metric = mean(overall_hitting))
twen9_hitting <- Lahman%>%filter(yearID==2019)
data$yr2hitting <- twen9_hitting$overall_hitting_metric
twen20_hitting <- Lahman%>%filter(yearID==2020)
data$yr1hitting <- twen20_hitting$overall_hitting_metric
twen8_hitting <- Lahman%>%filter(yearID==2018)
data$yr3hitting <- twen8_hitting$overall_hitting_metric

`summarise()` has grouped output by 'playerID'. You can override using the
`.groups` argument.


In [10]:
people_db = Lahman::People
people_db$firstlast = paste(Lahman::People$nameFirst, Lahman::People$nameLast, sep=" ")

## Join People data to OPS yrs

In [12]:
OPS_names <- left_join(data, people_db, by = c("bbref_id" = "bbrefID"))

## Data Cleaning

In [15]:
OPS_names$firstlast[OPS_names$firstlast == "J. D. Martinez"] <- "J.D. Martinez"
OPS_names$firstlast[OPS_names$firstlast == "J. T. Realmuto"] <- "J.T. Realmuto"
OPS_names$firstlast[OPS_names$firstlast == "AJ Pollock"] <- "A.J. Pollock"
OPS_names$firstlast[OPS_names$firstlast == "Jackie Bradley"] <- "Jackie Bradley Jr."

## Join with Batted Ball Tracking Data - Full Dataset

In [16]:
train_names <- OPS_names$firstlast

In [17]:
fangraphstrain <- fg_batter_leaders(startseason = 2020, endseason = 2020, qual = 0)%>%filter(PlayerNameRoute %in% train_names)

In [18]:
fangraphstrain <- fangraphstrain %>% rename("Oswing_pct" = "O-Swing_pct")

In [19]:
fangraphstrain <- fangraphstrain %>% rename("Zswing_pct" = "Z-Swing_pct")

In [20]:
fangraphstrain <- fangraphstrain%>%select(PlayerNameRoute, LD_pct, FB_pct, HR_FB, Pull_pct, Oswing_pct, Zswing_pct, EV, LA, HardHit_pct, Barrel_pct)

In [22]:
fulltrain <- left_join(OPS_names, fangraphstrain, by = c("firstlast" = "PlayerNameRoute"))

In [24]:
fulltrain <- fulltrain%>%select(firstlast, bbref_id, OPSY1, OPSY2, OPSY3, ops_pred, ops_real, yr1hitting, yr2hitting, yr3hitting, LD_pct, FB_pct, Pull_pct, Oswing_pct, Zswing_pct, EV, LA, HardHit_pct, Barrel_pct)

## Create Train and Test set, Train Models

In [27]:
#make this example reproducible
set.seed(1)

#use 70% of dataset as training set and 30% as test set
sample <- sample(c(TRUE, FALSE), nrow(fulltrain), replace=TRUE, prob=c(0.7,0.3))
train  <- fulltrain[sample, ]
test   <- fulltrain[!sample, ]

In [28]:
summary(lm(ops_real ~ OPSY1 + ops_pred + OPSY2 + OPSY3 + yr3hitting + yr2hitting+ yr1hitting + LD_pct +FB_pct + Pull_pct + Oswing_pct + Zswing_pct + EV + LA + HardHit_pct + Barrel_pct, data=train[complete.cases(train),]))


Call:
lm(formula = ops_real ~ OPSY1 + ops_pred + OPSY2 + OPSY3 + yr3hitting + 
    yr2hitting + yr1hitting + LD_pct + FB_pct + Pull_pct + Oswing_pct + 
    Zswing_pct + EV + LA + HardHit_pct + Barrel_pct, data = train[complete.cases(train), 
    ])

Residuals:
     Min       1Q   Median       3Q      Max 
-0.14677 -0.04932 -0.00904  0.05952  0.20900 

Coefficients:
              Estimate Std. Error t value Pr(>|t|)  
(Intercept) -0.3078536  0.7902308  -0.390   0.6978  
OPSY1        0.2886535  0.2184163   1.322   0.1896  
ops_pred    -0.1933934  0.8467805  -0.228   0.8199  
OPSY2        0.3946083  0.1846137   2.137   0.0352 *
OPSY3        0.0892047  0.1633714   0.546   0.5864  
yr3hitting   0.1074932  0.1619461   0.664   0.5085  
yr2hitting  -0.1207736  0.1979379  -0.610   0.5433  
yr1hitting   0.0050648  0.1613874   0.031   0.9750  
LD_pct      -0.1218091  0.2839701  -0.429   0.6690  
FB_pct       0.1018880  0.3819048   0.267   0.7902  
Pull_pct    -0.0351116  0.1398056  -0.251   0.80

In [29]:
summary(lm(ops_real ~ ops_pred + yr1hitting + EV + HardHit_pct + Barrel_pct, data=train[complete.cases(train),]))


Call:
lm(formula = ops_real ~ ops_pred + yr1hitting + EV + HardHit_pct + 
    Barrel_pct, data = train[complete.cases(train), ])

Residuals:
      Min        1Q    Median        3Q       Max 
-0.146611 -0.049261 -0.008634  0.050985  0.187462 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) -0.703942   0.598445  -1.176    0.242    
ops_pred     1.183097   0.280862   4.212 5.46e-05 ***
yr1hitting   0.005649   0.100472   0.056    0.955    
EV           0.007202   0.007438   0.968    0.335    
HardHit_pct -0.267861   0.227911  -1.175    0.243    
Barrel_pct   0.389618   0.297040   1.312    0.193    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.07732 on 102 degrees of freedom
Multiple R-squared:  0.3483,	Adjusted R-squared:  0.3163 
F-statistic:  10.9 on 5 and 102 DF,  p-value: 2.024e-08


## Test RMSE for Model Options

In [30]:
model <- lm(ops_real ~ OPSY1 + ops_pred + yr3hitting + OPSY2 + OPSY3 + LD_pct +FB_pct + Pull_pct + Oswing_pct + Zswing_pct + EV + LA + HardHit_pct + Barrel_pct, data=train[complete.cases(train),])
test_preds <- predict(model, test[complete.cases(test),])
test_obs <- test[complete.cases(test), 'ops_real']
our_rmse = sqrt(mean(((test_preds-test_obs)**2)$ops_real))
our_rmse

[1] 0.1108342

In [31]:
model2 <- lm(ops_real ~ ops_pred + yr1hitting + EV + HardHit_pct + Barrel_pct, data=train[complete.cases(train),])
test_preds <- predict(model2, test[complete.cases(test),])
test_obs <- test[complete.cases(test), 'ops_real']
our_rmse = sqrt(mean(((test_preds-test_obs)**2)$ops_real))
our_rmse

[1] 0.1117014

use fangraphs to pull hitting metric multiple years. left join that with the statcast data 2025. 
pivot the ops data to have ops years as columns. use fangraphs to have list of sox players
create regressed rate and age adj ops for sox players. 
merge the fangraphs and statcast data for the test set

## Creating Dataset for Sox 2026 Preds

In [33]:
fangraphstest <- fg_batter_leaders(startseason = 2025, endseason = 2025, qual = 100)%>% filter(team_name == "BOS")
#%>%select(Season, PlayerName, H, SF, BB, IBB, HBP, SO, AB, PA, OPS, LD_pct, FB_pct, HR_FB, Pull_pct, O-Swing_pct, Z-Swing_pct, EV, LA, HardHit_pct, Barrel_pct)

In [34]:
test_names <- fangraphstest$PlayerName

In [35]:
fangraphstest2 <- fg_batter_leaders(startseason = 2024, endseason = 2024, qual = 100)%>% filter(PlayerName %in% test_names)

In [36]:
fangraphstest3 <- fg_batter_leaders(startseason = 2023, endseason = 2023, qual = 100)%>% filter(PlayerName %in% test_names)

In [37]:
fangraphstest <- rbind(fangraphstest, fangraphstest2, fangraphstest3, fill = TRUE)

In [38]:
fangraphstest <- fangraphstest %>% rename("Oswing_pct" = "O-Swing_pct")

In [39]:
fangraphstest <- fangraphstest %>% rename("Zswing_pct" = "Z-Swing_pct")

In [40]:
fangraphstest <- fangraphstest%>%select(Season, PlayerName, Age, H, SF, BB, IBB, HBP, SO, AB, PA, OPS, LD_pct, FB_pct, Pull_pct, Oswing_pct, Zswing_pct, EV, LA, HardHit_pct, Barrel_pct)

In [42]:
# Vector of known suffixes (can be expanded)
suffixes <- c("Jr.", "Sr.", "II", "III", "IV", "V")

# Function to convert "First Middle Last Suffix" → "Last Suffix, First Middle"
convert_name_suffix <- function(full_name) {
  parts <- strsplit(trimws(full_name), "\\s+")
  
  sapply(parts, function(p) {
    n <- length(p)
    
    if (n < 2) {
      return(p)  # Single-word name, return as is
    }
    
    # Check if last word is a suffix
    if (p[n] %in% suffixes) {
      last <- paste(p[n-1], p[n])  # Last name + suffix
      first <- paste(p[1:(n-2)], collapse = " ")
    } else {
      last <- p[n]  # Just last name
      first <- paste(p[1:(n-1)], collapse = " ")
    }
    
    paste0(last, ", ", first)
  }, USE.NAMES = FALSE)
}

# Example usage
names <- fangraphstest$PlayerName

converted <- convert_name_suffix(names)
print(converted)
fangraphstest$lastname_firstname <- converted

 [1] "Duran, Jarren"      "Rafaela, Ceddanne"  "Bregman, Alex"     
 [4] "Story, Trevor"      "Anthony, Roman"     "Narváez, Carlos"   
 [7] "Abreu, Wilyer"      "Gonzalez, Romy"     "Refsnyder, Rob"    
[10] "Mayer, Marcelo"     "Hamilton, David"    "Yoshida, Masataka" 
[13] "Sogard, Nick"       "Campbell, Kristian" "Casas, Triston"    
[16] "Toro, Abraham"      "Wong, Connor"       "Duran, Jarren"     
[19] "Bregman, Alex"      "Abreu, Wilyer"      "Hamilton, David"   
[22] "Refsnyder, Rob"     "Wong, Connor"       "Rafaela, Ceddanne" 
[25] "Yoshida, Masataka"  "Toro, Abraham"      "Story, Trevor"     
[28] "Casas, Triston"     "Gonzalez, Romy"     "Bregman, Alex"     
[31] "Duran, Jarren"      "Casas, Triston"     "Yoshida, Masataka" 
[34] "Refsnyder, Rob"     "Story, Trevor"      "Wong, Connor"      


In [43]:
fangraphstest <- fangraphstest%>%mutate(overall_hitting = ((H + SF + BB +  IBB + HBP)/AB) - (SO/AB))

In [44]:
fangraphstest<- fangraphstest%>%pivot_wider(names_from = Season, values_from = c(PA, overall_hitting, OPS))

In [47]:
fangraphstest <- fangraphstest%>%select(PlayerName, lastname_firstname, Age, PA_2025,PA_2024,PA_2023,overall_hitting_2025, OPS_2025,OPS_2024,OPS_2023, LD_pct, FB_pct, Pull_pct, Oswing_pct, Zswing_pct, EV, LA, HardHit_pct, Barrel_pct)

In [48]:
fangraphstest <- fangraphstest%>%group_by(lastname_firstname)%>%summarise(across(where(is.numeric), max, na.rm = TRUE), .groups = "drop")
fangraphstest[fangraphstest == -Inf] <- NA

Warning message:
“There were 31 warnings in `summarise()`.
The first warning was:
ℹ In argument: `across(where(is.numeric), max, na.rm = TRUE)`.
ℹ In group 1: `lastname_firstname = "Abreu, Wilyer"`.
Caused by warning:
! The `...` argument of `across()` is deprecated as of dplyr 1.1.0.
Supply arguments directly to `.fns` through an anonymous function instead.

  # Previously
  across(a:b, mean, na.rm = TRUE)

  # Now
  across(a:b, \(x) mean(x, na.rm = TRUE))
ℹ Run `dplyr::last_dplyr_warnings()` to see the 30 remaining warnings.”


In [50]:
regressedOPSfinal <- c()
for(player in fangraphstest$lastname_firstname) {
    playerdf <- fangraphstest%>%filter(lastname_firstname == player)
    if(is.na(playerdf$PA_2023) && is.na(playerdf$PA_2024)){
        player_rate <- playerdf$OPS_2025
        reliability <- playerdf$PA_2025/((playerdf$PA_2025) + 400)
        regressed_rate <- (reliability * player_rate) + ((1 - reliability) * .719)
        regressedOPSfinal <- c(regressedOPSfinal, regressed_rate)

    } else if(is.na(playerdf$PA_2023)){
        player_rate <- ((playerdf$OPS_2024*4) + (playerdf$OPS_2025*5))/9
        reliability <- (playerdf$PA_2024 + playerdf$PA_2025)/((playerdf$PA_2024 + playerdf$PA_2025) + 800)
        regressed_rate <- (reliability * player_rate) + ((1 - reliability) * .719)
        regressedOPSfinal <- c(regressedOPSfinal, regressed_rate)
    } else if(is.na(playerdf$PA_2024)){
        player_rate <- ((playerdf$OPS_2023*4) + (playerdf$OPS_2025*5))/9
        reliability <- (playerdf$PA_2023 + playerdf$PA_2025)/((playerdf$PA_2023 + playerdf$PA_2025) + 800)
        regressed_rate <- (reliability * player_rate) + ((1 - reliability) * .719)
        regressedOPSfinal <- c(regressedOPSfinal, regressed_rate)
    } else {
        player_rate <- ((playerdf$OPS_2023*3) + (playerdf$OPS_2024*4) + (playerdf$OPS_2025*5))/12
        reliability <- (playerdf$PA_2023 + playerdf$PA_2024 + playerdf$PA_2025)/((playerdf$PA_2023 + playerdf$PA_2024 + playerdf$PA_2025) + 1200)
        regressed_rate <- (reliability * player_rate) + ((1 - reliability) * .719)
        regressedOPSfinal <- c(regressedOPSfinal, regressed_rate)
    }
}
regressedOPSfinal

[1] 0.7527327 0.7793845 0.7677224 0.6973412 0.7203124 0.7720821 0.7439653
 [8] 0.6873023 0.7074806 0.7225322 0.7010340 0.7489106 0.7070524 0.7083994
[15] 0.6890663 0.6765993 0.7297350

In [51]:
fangraphstest$regressed_OPS <- regressedOPSfinal

In [52]:
fangraphstest <- fangraphstest%>%mutate(ops_pred = ifelse(Age < 29, (1 + (29 - Age)*.006)*regressed_OPS, (1 + (Age - 29)*(-.003))*regressed_OPS))

In [54]:
colnames(fangraphstest) <- c("lastname_firstname", "Age", "PAY1","PAY2","PAY3","yr1hitting", "OPSY1","OPSY2","OPSY3", "LD_pct", "FB_pct", "Pull_pct", "Oswing_pct", "Zswing_pct", "EV", "LA", "HardHit_pct", "Barrel_pct", "regressedOPS", "ops_pred")

## Use Model2 to Make 2026 OPS Predictions for Sox

In [57]:
sox_preds <- predict(model2, fangraphstest)
fangraphstest$preds2026 <- sox_preds

lastname_firstname,Age,PAY1,PAY2,PAY3,yr1hitting,OPSY1,OPSY2,OPSY3,LD_pct,⋯,Pull_pct,Oswing_pct,Zswing_pct,EV,LA,HardHit_pct,Barrel_pct,regressedOPS,ops_pred,preds2026
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
"Abreu, Wilyer",26,417,447,NA,0.10455764,0.7865,0.7808,NA,0.1900,⋯,0.4529,0.3288,0.7538,91.5727,23.0364,0.5018,0.1232,0.7527327,0.7662819,0.7763256
"Anthony, Roman",21,303,NA,NA,0.14396887,0.8591,NA,NA,0.1782,⋯,0.4310,0.2257,0.5764,94.4688,6.8960,0.6034,0.1552,0.7793845,0.8167950,0.8424209
"Bregman, Alex",31,495,634,724,0.26096998,0.8215,0.7681,0.8038,0.1862,⋯,0.4740,0.2654,0.6912,90.1403,18.1566,0.4438,0.0658,0.7677224,0.7631161,0.7563196
"Campbell, Kristian",23,263,NA,NA,0.05676856,0.6644,NA,NA,0.1847,⋯,0.2866,0.2590,0.6348,88.6327,4.6624,0.4177,0.0506,0.6973412,0.7224455,0.6972600
"Casas, Triston",25,112,243,502,0.06060606,0.5798,0.7997,0.8560,0.2098,⋯,0.3889,0.2814,0.7539,91.0895,15.6787,0.4656,0.1333,0.7203124,0.7375999,0.7522955
"Duran, Jarren",28,696,735,362,0.10645161,0.7743,0.8338,0.8282,0.2562,⋯,0.3824,0.3466,0.7419,91.7625,12.0221,0.4681,0.0967,0.7720821,0.7767146,0.7887482
"Gonzalez, Romy",28,341,216,NA,0.13333333,0.8256,0.7226,NA,0.2008,⋯,0.3180,0.4037,0.7748,93.3050,5.4664,0.5732,0.1255,0.7439653,0.7484291,0.7496135
"Hamilton, David",27,194,317,NA,0.01129944,0.5899,0.6974,NA,0.2057,⋯,0.4286,0.2807,0.6573,87.0077,15.8134,0.3224,0.0602,0.6873023,0.6955500,0.6827474
"Mayer, Marcelo",22,136,NA,NA,-0.02362205,0.6736,NA,NA,0.1724,⋯,0.3793,0.3232,0.7045,90.0069,6.6782,0.5172,0.0920,0.7074806,0.7371948,0.7136307


In [61]:
fangraphstest <- fangraphstest%>%select(lastname_firstname, Age, preds2026)

In [60]:
getwd()

[1] "/home/jovyan/Player Projections"

In [62]:
write.csv(fangraphstest, file = "/home/jovyan/Player Projections/soxpreds26.csv", row.names = FALSE)